# Zbieranie korpusu kolokacji — Colab GPU

Parsuje polskie teksty Stanzą i zapisuje trójki kolokacyjne do pliku.

**Zanim uruchomisz:** Środowisko wykonawcze → Zmień typ środowiska → **GPU (T4)**.

Uruchamiaj komórki po kolei. Komórka **3b** sprawdza środowisko — jeśli coś się
nie doinstalowało, dowiesz się tam, a nie w połowie zbierania.

Zapis idzie domyślnie **lokalnie**, bez Dysku Google. Dysk wymaga interaktywnej
autoryzacji (okienko wyboru konta) i przy przebiegu próbnym jest zbędnym
punktem awarii; przy pełnej skali włącz go w komórce 4.

Kod pochodzi z repozytorium projektu, a nie jest wklejony do notebooka — celowo.
Ekstraktor trójek musi być **dokładnie ten sam**, którego użyje dodatek w czasie
pisania. Rozjazd logiki klucza między budową bazy a zapytaniem nie objawiłby się
żadnym błędem — tylko cicho zerową skutecznością podpowiedzi.

In [ ]:
# 1. Kontrola GPU — bez niego przebieg potrwa kilkanaście razy dłużej
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "BRAK GPU — zmień typ środowiska wykonawczego"

In [ ]:
# 2. Kod projektu + zależności
#
# Repozytorium jest publiczne, więc klon idzie anonimowo i nic nie trzeba
# konfigurować.
#
# Gdyby kiedyś wróciło do prywatnego: utwórz fine-grained token na GitHubie
# (Settings → Developer settings → Personal access tokens; dostęp tylko do tego
# repo, uprawnienie Contents: Read-only), a w Colabie dodaj go pod ikoną klucza
# (🔑) jako sekret o nazwie GH_TOKEN. Kod poniżej wykryje go sam.

WLASCICIEL = "SernikNaZimno"
NAZWA_REPO = "Autokorekta-Kolokacji"

import os, shutil, subprocess

token = None
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception:
    pass  # brak sekretu — repo jest publiczne, więc to normalna ścieżka

url = (
    f"https://{token}@github.com/{WLASCICIEL}/{NAZWA_REPO}.git"
    if token
    else f"https://github.com/{WLASCICIEL}/{NAZWA_REPO}.git"
)

if os.path.exists("/content/projekt"):
    shutil.rmtree("/content/projekt")

# Bez tego git czeka na hasło w nieskończoność zamiast zwrócić błąd.
os.environ["GIT_TERMINAL_PROMPT"] = "0"
wynik = subprocess.run(
    ["git", "clone", "--depth", "1", url, "/content/projekt"],
    capture_output=True, text=True,
)

# Zatrzymujemy się TUTAJ, jeśli klon padł. Wcześniejsza wersja leciała dalej,
# przez co brak repozytorium ujawniał się dopiero jako „No module named
# 'backend'" sześć komórek później i wyglądał na błąd kodu, a nie dostępu.
if wynik.returncode != 0:
    komunikat = wynik.stderr.replace(token, "***") if token else wynik.stderr
    raise RuntimeError(
        "KLON NIEUDANY — notebook zatrzymany celowo.\n\n"
        + komunikat.strip()
        + "\n\nJeśli widzisz 'could not read Password' lub 'Repository not found':\n"
        "repozytorium nie jest publiczne. Sprawdź GitHub → Settings → General →\n"
        "Change repository visibility, albo dodaj sekret GH_TOKEN (patrz góra komórki)."
    )

%cd /content/projekt
print("sklonowano OK")
!pip install -q stanza datasets morfeusz2

In [ ]:
# 3. Model polski (pobiera się raz, ~500 MB)
import stanza
stanza.download("pl", verbose=False)
print("model gotowy")

In [ ]:
# 3b. KONTROLA ŚRODOWISKA — uruchom przed zbieraniem
#
# `pip install -q` NIE przerywa notebooka, gdy instalacja się nie uda. Bez tej
# komórki brakujący pakiet objawiłby się dopiero w środku zbierania i wyglądał
# na błąd pipeline'u, a nie środowiska.

import sys, os

KORZEN = "/content/projekt"   # ścieżka bezwzględna: nie zależy od katalogu roboczego
if KORZEN not in sys.path:
    sys.path.insert(0, KORZEN)

problemy = []

if not os.path.isdir(os.path.join(KORZEN, "backend")):
    problemy.append(f"brak {KORZEN}/backend — klon repozytorium się nie powiódł (komórka 2)")

try:
    import torch
    print(f"torch {torch.__version__}   CUDA dostępna: {torch.cuda.is_available()}")
    if not torch.cuda.is_available():
        problemy.append("BRAK GPU — Środowisko wykonawcze → Zmień typ środowiska → T4")
except Exception as e:
    problemy.append(f"torch: {e}")

for nazwa in ["stanza", "datasets", "morfeusz2"]:
    try:
        __import__(nazwa)
        print(f"{nazwa}: OK")
    except Exception as e:
        problemy.append(f"{nazwa}: {e}")

for modul in ["backend.ekstraktor", "backend.baza", "backend.czyszczenie",
              "backend.slownik", "backend.pipeline"]:
    try:
        __import__(modul)
        print(f"{modul}: OK")
    except Exception as e:
        problemy.append(f"{modul}: {e}")

# Morfeusz musi nie tylko się zaimportować, ale i mieć wczytany słownik
try:
    from backend.slownik import WalidatorSlownikowy
    w = WalidatorSlownikowy()
    assert w.znane("decyzja") and not w.znane("złdo")
    print("filtr słownikowy: OK")
except Exception as e:
    problemy.append(f"filtr słownikowy: {e}")

print()
if problemy:
    print("PROBLEMY — napraw przed dalszymi krokami:")
    for p in problemy:
        print(f"  · {p}")
else:
    print("Wszystko gotowe.")

In [ ]:
# 4. Gdzie zapisać wynik
#
# Dysk Google wymaga INTERAKTYWNEJ autoryzacji — wyskakuje okienko, w którym
# trzeba wybrać konto i potwierdzić. Jeśli je przeoczysz albo zamkniesz,
# komórka po prostu czeka i cały notebook staje.
#
# Dla przebiegu próbnego (~15 min) prościej zapisać lokalnie i pobrać plik
# na końcu — sesja tyle wytrzyma bez problemu.
# Przy pełnej skali (kilka godzin) ustaw UZYJ_DYSKU = True, bo rozłączenie
# sesji skasowałoby wynik wielogodzinnego przebiegu.

UZYJ_DYSKU = False

from pathlib import Path

if UZYJ_DYSKU:
    from google.colab import drive
    drive.mount("/content/drive")
    KATALOG = Path("/content/drive/MyDrive/kolokacje")
else:
    KATALOG = Path("/content/wyniki")

KATALOG.mkdir(parents=True, exist_ok=True)
print(f"wyniki trafią do: {KATALOG}")

In [ ]:
# 5. Zbieranie
#
# Budżet jest PER ŹRÓDŁO. Bez podziału Wikipedia wypełniłaby cały limit —
# strumieniuje się szybciej niż C4, a chcemy oba rejestry.
#
# Przebieg próbny: 5 mln tokenów (~15 min na T4).
# Pełna skala: podnieś do 60/40 mln, ustaw UZYJ_DYSKU = True i licz kilka godzin.

import sys
if "/content/projekt" not in sys.path:
    sys.path.insert(0, "/content/projekt")

from backend.pipeline import zbierz

BUDZET = {"wiki": 3_000_000, "web": 2_000_000}
WYJSCIE = KATALOG / "trojki_5M.tsv.gz"

stat = zbierz(BUDZET, WYJSCIE, gpu=True, partia=256)
stat

In [ ]:
# 6. Baza + pierwsze spojrzenie na sygnał
from backend.baza import BazaKolokacji, zbuduj
from backend.pipeline import czytaj_trojki

BAZA = KATALOG / "kolokacje_5M.sqlite"
print(zbuduj(czytaj_trojki(WYJSCIE), BAZA, min_pary=3))

with BazaKolokacji(BAZA) as db:
    print(db.statystyki())
    print(f"udział wiki {db.udzial_zrodla('wiki'):.1%}, web {db.udzial_zrodla('web'):.1%}")
    print()
    print("Czasowniki łączące się z 'decyzja' (obj:acc):")
    for k in db.alternatywy("obj:acc", "decyzja", limit=10):
        print(f"  {k.lemat:<16} f={k.f:<4} logDice={k.logdice:.2f}")
    print()
    f = db.czestosc_slotu_dep("obj:acc", "decyzja")
    print(f"częstość brzegowa 'decyzja' w obj:acc = {f}")
    print("slot zbadany" if db.slot_zbadany("obj:acc", "decyzja") else "slot NIEzbadany — silnik milczy")

In [ ]:
# 7. Kontrola skażenia domenowego
#
# Para o wysokiej częstości pochodząca wyłącznie z webu to zwykle boilerplate
# (stopka, klauzula, szablon ogłoszenia), a nie norma językowa.

with BazaKolokacji(BAZA) as db:
    print("Pary wyłącznie z webu, najczęstsze:")
    for h, s, d, f in db.con.execute(
        """SELECT p.head, p.slot, p.dep, p.f FROM pary p
           WHERE p.f >= 10 AND NOT EXISTS (
             SELECT 1 FROM pary_zrodlo z WHERE z.head=p.head AND z.slot=p.slot
             AND z.dep=p.dep AND z.zrodlo='wiki')
           ORDER BY p.f DESC LIMIT 20"""):
        print(f"  {h} --{s}--> {d}   f={f}")

In [ ]:
# 8. Pobranie wyników na dysk lokalny
#
# Potrzebne tylko przy UZYJ_DYSKU = False. Colab kasuje pliki po rozłączeniu
# sesji, więc bazę trzeba ściągnąć, zanim to nastąpi.

if not UZYJ_DYSKU:
    from google.colab import files
    for plik in [BAZA, WYJSCIE]:
        print(f"{plik.name}: {plik.stat().st_size / 1024 / 1024:.1f} MB")
        files.download(str(plik))
else:
    print(f"Pliki są już na Dysku: {KATALOG}")